# Hackathón - Modelos Predictivos de Riesgo de Desplazamiento por Desastres


# Contexto

# Importaciones

In [ ]:
import os, zipfile

# Análisis de datos
import numpy as np
import pandas as pd

# Modelamiento
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import roc_auc_score

# 1. Carga de los datos

Se descargan los datos correspondientes desde la fuente dispuesta

In [ ]:
URL_DATOS =   # Por políticas del PNUD los datos no pueden ser publicados
DATA_DIR = os.environ.get("HACKATHON_DATA", "datos")

if not os.path.exists(os.path.join(DATA_DIR, "entrenamiento.csv")):
    if URL_DATOS:
        import gdown
        gdown.download(URL_DATOS, "hackathon_datos.zip", quiet=False, fuzzy=True)
    with zipfile.ZipFile("hackathon_datos.zip") as z:
        z.extractall(DATA_DIR)

leer = lambda f: pd.read_csv(os.path.join(DATA_DIR, f), dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
train = leer("entrenamiento.csv")
test_pub = leer("prueba_equipos.csv")     # periodo de la tabla de posiciones en vivo
test_priv = leer("prueba_oculta.csv")     # periodo de la nota oficial
dic = pd.read_csv(os.path.join(DATA_DIR, "diccionario_datos.csv"))
print(train.shape, test_pub.shape, test_priv.shape)

(62814, 25) (20938, 13) (22040, 13)


# 2. Generalidades

Se identifican las variables, objetivos y generalidades del dataset

In [ ]:
pd.set_option("display.max_colwidth", 120)
display(dic[["columna", "archivos", "grupo", "descripcion"]])
print("Prevalencia de tiene_evento:", round(train.tiene_evento.mean(), 3))
print("Meses con evento por año:")
display(train.groupby("anio").tiene_evento.mean().round(3).to_frame())
print("Personas desplazadas cuando hay evento:")
display(train.loc[train.tiene_evento == 1, "personas_desplazadas"].describe().round(1).to_frame())

Además, se construye el historial de entrenamiento para evitar que exista fuga de información, garantizando que cada registro y predicción se realice solo con los datos disponibles hasta ese periodo de tiempo

In [ ]:
def historial_entrenamiento(df):
    """Historial estrictamente previo a cada fila (para entrenar)."""
    df = df.sort_values(["DIVIPOLA", "fecha"]).copy()
    df["logp"] = np.log1p(df["personas_desplazadas"])
    g = df.groupby("DIVIPOLA")
    df["hist_tasa"] = g["tiene_evento"].transform(lambda s: s.shift(1).expanding().mean())
    df["hist_logp"] = g["logp"].transform(lambda s: s.shift(1).expanding().mean())
    df["hist_tasa_mes"] = df.groupby(["DIVIPOLA", "mes"])["tiene_evento"].transform(lambda s: s.shift(1).expanding().mean())
    return df

def historial_congelado(hist, filas):
    """Historial calculado con todo `hist`, aplicado a filas posteriores (para predecir)."""
    hist = hist.assign(logp=np.log1p(hist["personas_desplazadas"]))
    h = hist.groupby("DIVIPOLA").agg(hist_tasa=("tiene_evento", "mean"), hist_logp=("logp", "mean")).reset_index()
    hm = hist.groupby(["DIVIPOLA", "mes"])["tiene_evento"].mean().rename("hist_tasa_mes").reset_index()
    return filas.merge(h, on="DIVIPOLA", how="left").merge(hm, on=["DIVIPOLA", "mes"], how="left")

# 3. Ingeniería de variables

Para el modelamiento se realizaron las siguientes transformaciones:
- Se aplicó una transformación log1p a densidad poblacional, ingresos tributarios e inversión en gestión del riesgo, debido a su alta asimetría.
- La variable mes se transformó mediante seno y coseno para representar su comportamiento cíclico anual.

In [ ]:
def features(df):
    X = pd.DataFrame(index=df.index)
    for c in ["densidad_poblacional", "ingresos_tributarios", "inversion_gestion_riesgo"]:
        X["log_" + c] = np.log1p(df[c])                      # variables muy sesgadas
    for c in ["nbi_pct", "elevacion_media", "pendiente_media", "pct_pendiente_alta", "ONI",
              "hist_tasa", "hist_tasa_mes", "hist_logp"]:
        X[c] = df[c]
    X["mes_sin"], X["mes_cos"] = np.sin(2 * np.pi * df["mes"] / 12), np.cos(2 * np.pi * df["mes"] / 12)
    X = X.drop(columns=[c for c in VARIABLES_FUERA if c in X.columns])
    return X

def entrenar(df_hist):
    d = df_hist[df_hist["anio"] >= 2019]                     # 2018 sin historial previo: solo calentamiento
    modelo = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))
    modelo.fit(features(d), d["tiene_evento"])
    return modelo

def predecir(modelo, filas):
    p = filas[["DIVIPOLA", "fecha"]].copy()
    p["prob_evento"] = modelo.predict_proba(features(filas))[:, 1]
    # magnitud: la métrica es en escala log1p, así que estimamos ahí:
    # log1p(personas) esperado ≈ probabilidad × tamaño típico (en log) de los eventos del municipio. ¡Mejórenlo!
    log_tipico = (filas["hist_logp"] / filas["hist_tasa"].clip(lower=0.01)).fillna(0).to_numpy()
    p["personas_desplazadas_estimadas"] = np.expm1(p["prob_evento"] * log_tipico)
    # intervalo del 80%: de 0 hasta el tamaño típico, solo donde el riesgo es apreciable
    p["personas_desplazadas_q10"] = 0.0
    p["personas_desplazadas_q90"] = np.where(p["prob_evento"] >= 0.15, np.expm1(log_tipico), 0.0)
    return p

Adicionalmente, se incorporaron otras variables como:
- Del Censo Nacional de Vivienda (2018) se incorporaron variables como número total de viviendas (en logaritmo dada su asimetría), tipo de paredes, tipo de suelos y tipo de viviendas entre otras (en porcentaje para evitar el sesgo producido por escalas absolutas).
- Se obtuvo el tipo de bioma al que pertenece cada municipio según los datos de la WWF, permitiendo identificar zonas que pueden llegar a ser más susceptibles a desastres y desplazamientos según sus condiciones y vegetación
- Se incluyeron índices meteorológicos promediados desde 1990 hasta 2020

## Viviendas

In [ ]:
RUTA_VIVIENDAS = "/content/Viviendas.xlsx"

viv_raw = pd.read_excel(RUTA_VIVIENDAS, dtype={"DIVIPOLA": str})
viv_raw["DIVIPOLA"] = viv_raw["DIVIPOLA"].str.zfill(5)

col_tipo = ["Casa", "Apartamento", "Tipo cuarto", "Vivienda tradicional indígena", "Vivienda tradicional étnica"]
col_paredes_precarias = ["Tapia pisada, bahareque, adobe", "Madera burda, tabla, tablón",
                         "Caña, esterilla, otros vegetales",
                         "Materiales de desecho (Zinc, tela, cartón, latas, plásticos, otros)", "Sin paredes"]
col_paredes = col_paredes_precarias + ["Bloque, ladrillo, piedra, madera pulida", "Concreto vaciado",
                                       "Material prefabricado", "Guadua"]

total = viv_raw[col_tipo].sum(axis=1).clip(lower=1)
total_paredes = viv_raw[col_paredes].sum(axis=1).clip(lower=1)
VIVIENDAS = pd.DataFrame({
    "log_viviendas": np.log1p(total),
    "pct_apartamento": viv_raw["Apartamento"] / total,
    "pct_vivienda_tradicional": (viv_raw["Vivienda tradicional indígena"]
                                 + viv_raw["Vivienda tradicional étnica"]) / total,
    "pct_paredes_precarias": viv_raw[col_paredes_precarias].sum(axis=1) / total_paredes,
    "pct_piso_tierra": viv_raw["Tierra, arena, barro"] / total_paredes,
}).set_index(viv_raw["DIVIPOLA"])

cobertura = train["DIVIPOLA"].drop_duplicates().isin(VIVIENDAS.index).mean()
print(f"Municipios del reto con datos de vivienda: {cobertura:.1%}")
print(VIVIENDAS.describe().round(3).to_string())

Municipios del reto con datos de vivienda: 100.0%
       log_viviendas  pct_apartamento  pct_vivienda_tradicional  pct_paredes_precarias  pct_piso_tierra
count       1122.000         1122.000                  1122.000               1122.000         1122.000
mean           8.235            0.088                     0.019                  0.317            0.158
std            1.187            0.112                     0.076                  0.232            0.169
min            3.714            0.000                     0.000                  0.001            0.000
25%            7.509            0.017                     0.000                  0.127            0.030
50%            8.144            0.044                     0.000                  0.272            0.098
75%            8.849            0.115                     0.002                  0.458            0.228
max           14.667            0.808                     0.754                  0.990            0.875


## Biomas

In [ ]:
RUTA_BIOMAS = "Biomas.xlsx"

bio_raw = pd.read_excel(RUTA_BIOMAS, dtype=str)
bio_raw["DIVIPOLA"] = bio_raw["DIVIPOLA"].str.zfill(5)
BIOMAS = bio_raw.set_index("DIVIPOLA")[["region_natural", "bioma_wwf_es"]]
CATS_REGION = sorted(BIOMAS["region_natural"].dropna().unique())
CATS_BIOMA = sorted(BIOMAS["bioma_wwf_es"].dropna().unique())

cobertura = train["DIVIPOLA"].drop_duplicates().isin(BIOMAS.index).mean()
print(f"Municipios del reto con bioma/región: {cobertura:.1%}")
print(BIOMAS["bioma_wwf_es"].value_counts().to_string())

## Índices meteorológicos

In [ ]:
# Normales climatológicas IDEAM 1991-2020 por municipio y mes
RUTA_NORMALES = "/content/normales_climatologicas_municipio_mes_v2.csv"

norm_raw = pd.read_csv(RUTA_NORMALES, dtype={"DIVIPOLA": str})
norm_raw["DIVIPOLA"] = norm_raw["DIVIPOLA"].str.zfill(5)
COLS_NORMALES = ["precip_normal_mes", "precip_normal_anual", "precip_estacional", "dias_lluvia_normal_mes"]
NORMALES = norm_raw.set_index(norm_raw["DIVIPOLA"] + "_" + norm_raw["mes"].astype(str))[COLS_NORMALES]

def llave_normales(df):
    return df["DIVIPOLA"].astype(str).str.zfill(5) + "_" + df["mes"].astype(int).astype(str)

for nombre, d in [("entrenamiento", train), ("prueba_equipos", test_pub), ("prueba_oculta", test_priv)]:
    print(f"{nombre}: filas con normales {llave_normales(d).isin(NORMALES.index).mean():.1%}")
print(f"Municipios con estación propia: {norm_raw.drop_duplicates('DIVIPOLA')['con_estacion_propia'].mean():.1%}")

# 4. Modelamiento

In [ ]:
class ModeloRiesgo:
    """Contenedor de los submodelos entrenados."""
    pass


ALPHA = 60.0         # fuerza del shrinkage hacia el promedio nacional
ALPHA_SEV = 60.0     # shrinkage del tamaño típico de los eventos (separado del de la tasa)
UMBRAL_CERO = 0.03   # por debajo de esta prob., el intervalo colapsa a [0, 0]
PESO_LOGIT = 0.0     # peso de la logística en la mezcla (0 = solo LightGBM)
PESO_HURDLE = 1.0    # magnitud: 0 = solo regresor, 1 = solo probabilidad x tamaño típico

USAR_VIVIENDAS = True     # variables de vivienda del censo (requiere la celda de Viviendas.xlsx)
USAR_BIOMA = True         # bioma WWF del municipio como categoría (requiere la celda de Biomas.xlsx)
USAR_REGION = False       # región natural del municipio como categoría (requiere la celda de Biomas.xlsx)
USAR_LLUVIA = False       # lluvia mensual NASA POWER (requiere las celdas de lluvia)
USAR_NORMALES = True     # normales climatológicas IDEAM por municipio y mes (requiere su celda de carga)

VARIABLES_FUERA = ["nbi_pct", "ONI", "hist_logp", "cap_institucional",
                   "log_inversion_gestion_riesgo", "es_la_nina"]
SEMILLAS = [42, 7, 2024]
PARAMS_CLF = dict(n_estimators=700, learning_rate=0.02, num_leaves=15, min_child_samples=150,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, verbose=-1)
PARAMS_REG = dict(n_estimators=500, learning_rate=0.02, num_leaves=15, min_child_samples=150,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, verbose=-1)


# 1. PERFIL MUNICIPAL (sin fuga, con shrinkage bayesiano)
def _perfil_expansivo(df):
    """Perfil por fila usando SOLO meses anteriores del mismo municipio (sin fuga)."""
    d = df.sort_values(["DIVIPOLA", "fecha"])
    g = d.groupby("DIVIPOLA")
    p_glob = d["tiene_evento"].mean()
    z = np.log1p(d["personas_desplazadas"]) * d["tiene_evento"]
    sev_glob = z.sum() / max(d["tiene_evento"].sum(), 1)

    n_prev = g.cumcount()
    ev_prev = g["tiene_evento"].cumsum() - d["tiene_evento"]
    z_prev = z.groupby(d["DIVIPOLA"]).cumsum() - z

    out = pd.DataFrame(index=d.index)
    out["tasa_suave"] = (ev_prev + ALPHA * p_glob) / (n_prev + ALPHA)
    out["sev_cond"] = (z_prev + ALPHA_SEV * sev_glob) / (ev_prev + ALPHA_SEV)   # log1p(personas) | evento
    return out.reindex(df.index)


def _perfil_final(df):
    """Perfil con todo el histórico de entrenamiento, para predecir el futuro."""
    p_glob = df["tiene_evento"].mean()
    z = np.log1p(df["personas_desplazadas"]) * df["tiene_evento"]
    sev_glob = z.sum() / max(df["tiene_evento"].sum(), 1)
    g = pd.DataFrame({"DIVIPOLA": df["DIVIPOLA"], "ev": df["tiene_evento"], "z": z}).groupby("DIVIPOLA")
    s = g.sum()
    n = g.size()
    perfil = pd.DataFrame({
        "tasa_suave": (s["ev"] + ALPHA * p_glob) / (n + ALPHA),
        "sev_cond": (s["z"] + ALPHA_SEV * sev_glob) / (s["ev"] + ALPHA_SEV),
    })
    return perfil, {"tasa_suave": p_glob, "sev_cond": sev_glob}


# 2. FEATURES (las originales + las nuevas)
def features(df, deptos=None):
    X = pd.DataFrame(index=df.index)
    for c in ["densidad_poblacional", "ingresos_tributarios", "inversion_gestion_riesgo"]:
        X["log_" + c] = np.log1p(df[c])                      # variables muy sesgadas
    for c in ["nbi_pct", "elevacion_media", "pendiente_media", "pct_pendiente_alta", "ONI",
              "hist_tasa", "hist_tasa_mes", "hist_logp"]:
        X[c] = df[c]
    X["mes_sin"], X["mes_cos"] = np.sin(2 * np.pi * df["mes"] / 12), np.cos(2 * np.pi * df["mes"] / 12)

    # perfil municipal suavizado (se agrega en entrenar/predecir)
    for c in ["tasa_suave", "sev_cond"]:
        if c in df.columns:
            X[c] = df[c]

    # interacciones físico-socioeconómicas
    es_la_nina = (df["ONI"] <= -0.5).astype(float)
    lluvias = df["mes"].isin([4, 5, 10, 11]).astype(float)       # bimodal colombiana
    X["es_la_nina"] = es_la_nina
    X["es_temporada_lluvias"] = lluvias
    X["lluvia_x_nina"] = lluvias * es_la_nina * df["ONI"].abs()
    X["susc_masa"] = df["pct_pendiente_alta"] * df["nbi_pct"] * df["pendiente_media"]
    X["susc_inund"] = (1 / (np.log1p(df["elevacion_media"].clip(lower=0)) + 1)) \
                      * np.log1p(df["densidad_poblacional"]) * (1 + es_la_nina)
    X["cap_institucional"] = df["inversion_gestion_riesgo"] / (df["densidad_poblacional"] * 100 + 1)

    # vivienda por municipio (censo): exposición y vulnerabilidad física
    if USAR_VIVIENDAS:
        div = df["DIVIPOLA"].astype(str).str.zfill(5)
        for c in VIVIENDAS.columns:
            X[c] = div.map(VIVIENDAS[c]).to_numpy()

    # normales climatológicas IDEAM 1991-2020: lluvia típica de ese mes en ese municipio
    if USAR_NORMALES:
        llave_n = llave_normales(df)
        for c in COLS_NORMALES:
            X[c] = llave_n.map(NORMALES[c]).to_numpy()

    # bioma y región natural (categorías fijas para que coincidan en entrenamiento y prueba)
    if USAR_BIOMA:
        div = df["DIVIPOLA"].astype(str).str.zfill(5)
        X["bioma_cat"] = pd.Categorical(div.map(BIOMAS["bioma_wwf_es"]).to_numpy(), categories=CATS_BIOMA)
    if USAR_REGION:
        div = df["DIVIPOLA"].astype(str).str.zfill(5)
        X["region_cat"] = pd.Categorical(div.map(BIOMAS["region_natural"]).to_numpy(), categories=CATS_REGION)

    # departamento (2 primeros dígitos del DIVIPOLA) como categoría
    if deptos is not None:
        X["depto_cat"] = pd.Categorical(df["DIVIPOLA"].astype(str).str[:2], categories=deptos)

    # variables que no aportan según la importancia por permutación
    X = X.drop(columns=[c for c in VARIABLES_FUERA if c in X.columns])
    return X


# 3. ENTRENAMIENTO
def entrenar(df_hist):
    df_hist = df_hist.copy()
    df_hist[["tasa_suave", "sev_cond"]] = _perfil_expansivo(df_hist)

    d = df_hist[df_hist["anio"] >= 2019]                     # 2018 sin historial previo: solo calentamiento
    m = ModeloRiesgo()
    m.deptos = sorted(df_hist["DIVIPOLA"].astype(str).str[:2].unique())
    X, y = features(d, m.deptos), d["tiene_evento"]
    print(f"Entrenando con {X.shape[1]} variables")
    z = np.log1p(d["personas_desplazadas"])
    # baseline logístico (sin las categorías, la regresión logística no las acepta)
    m.logit = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                            LogisticRegression(max_iter=2000)).fit(X.select_dtypes(exclude="category"), y)
    # Submodelo 1: probabilidad de evento (promedio de varias semillas)
    m.clfs = [LGBMClassifier(objective="binary", random_state=s, **PARAMS_CLF).fit(X, y)
              for s in SEMILLAS]
    # Submodelo 2: magnitud en log1p (MSE en log1p = métrica del reto)
    m.reg = LGBMRegressor(objective="regression", random_state=42, **PARAMS_REG).fit(X, z)
    # Submodelo 3: cuantiles 10 / 90
    m.q10 = LGBMRegressor(objective="quantile", alpha=0.10, random_state=42, **PARAMS_REG).fit(X, z)
    m.q90 = LGBMRegressor(objective="quantile", alpha=0.90, random_state=42, **PARAMS_REG).fit(X, z)

    m.perfil, m.perfil_glob = _perfil_final(df_hist)
    return m


# 4. PREDICCIÓN
def predecir(modelo, filas):
    f = filas.copy()
    for c in ["tasa_suave", "sev_cond"]:
        f[c] = f["DIVIPOLA"].map(modelo.perfil[c]).fillna(modelo.perfil_glob[c])
    X = features(f, modelo.deptos)

    p = filas[["DIVIPOLA", "fecha"]].copy()
    # probabilidad: mezcla del baseline logístico y LightGBM
    prob = (PESO_LOGIT * modelo.logit.predict_proba(X.select_dtypes(exclude="category"))[:, 1]
            + (1 - PESO_LOGIT) * np.mean([c.predict_proba(X)[:, 1] for c in modelo.clfs], axis=0))
    p["prob_evento"] = np.clip(prob, 0, 1)

    # magnitud en escala log1p (la métrica es RMSE sobre log1p)
    z_reg = np.clip(modelo.reg.predict(X), 0, None)
    z_hurdle = p["prob_evento"].to_numpy() * f["sev_cond"].to_numpy()
    z_hat = PESO_HURDLE * z_hurdle + (1 - PESO_HURDLE) * z_reg
    # tamaño típico del municipio, usado como piso del intervalo (regla original)
    log_tipico = (filas["hist_logp"] / filas["hist_tasa"].clip(lower=0.01)).fillna(0).to_numpy()
    p["personas_desplazadas_estimadas"] = np.expm1(z_hat)

    # intervalo 80%
    q10 = np.clip(modelo.q10.predict(X), 0, None)
    q90 = np.clip(modelo.q90.predict(X), 0, None)
    q90 = np.maximum(q90, np.where(p["prob_evento"] >= 0.15, log_tipico, 0.0))   # regla original como piso
    q90 = np.maximum.reduce([q90, q10, z_hat])                                   # q10 <= estimado <= q90
    q10 = np.minimum(q10, z_hat)
    bajo = p["prob_evento"].to_numpy() < UMBRAL_CERO                              # riesgo ínfimo: [0, 0]
    q10, q90 = np.where(bajo, 0.0, q10), np.where(bajo, 0.0, q90)
    p.loc[bajo, "personas_desplazadas_estimadas"] = 0.0
    p["personas_desplazadas_q10"] = np.expm1(q10)
    p["personas_desplazadas_q90"] = np.expm1(q90)
    return p

## Validación del modelo

In [ ]:
def metricas(pred, verdad):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y = np.log1p(m["personas_desplazadas"])
    est = np.log1p(m["personas_desplazadas_estimadas"])
    lo, hi = np.log1p(m["personas_desplazadas_q10"]), np.log1p(m["personas_desplazadas_q90"])
    wink = (hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)   # alpha = 0.2
    return {"auc": roc_auc_score(m["tiene_evento"], m["prob_evento"]),
            "rmse_log": float(np.sqrt(np.mean((y - est) ** 2))), "winkler": float(wink.mean())}

def puntos(met, nula):
    """Parte automática de la nota (máx. 75 puntos), como la tabla de posiciones."""
    s_auc = max(0, (met["auc"] - 0.5) / 0.5)
    s_rmse = min(1, max(0, 1 - met["rmse_log"] / nula["rmse_log"]))
    s_wink = min(1, max(0, 1 - met["winkler"] / nula["winkler"]))
    return 100 * (0.50 * s_auc + 0.20 * s_rmse + 0.05 * s_wink)

CORTE = "2021-09-01"
tr, va = train[train.fecha <= CORTE], train[train.fecha > CORTE]
modelo = entrenar(historial_entrenamiento(tr))
print(tr.columns)
pred_va = predecir(modelo, historial_congelado(tr, va))

nula_pred = va[["DIVIPOLA", "fecha"]].assign(prob_evento=0.5, personas_desplazadas_estimadas=0.0,
                                             personas_desplazadas_q10=0.0, personas_desplazadas_q90=0.0)
nula = metricas(nula_pred, va)
met = metricas(pred_va, va)
print("Predicción nula:", {k: round(v, 3) for k, v in nula.items()})
print("Este modelo:    ", {k: round(v, 3) for k, v in met.items()})
print(f"Puntos automáticos en validación: {puntos(met, nula):.1f} / 75")

Entrenando con 24 variables
Index(['DIVIPOLA', 'fecha', 'anio', 'mes', 'tiene_evento', 'n_eventos',
       'personas_desplazadas', 'familias_desplazadas', 'viv_destruidas',
       'viv_averiadas', 'n_movimiento_masa', 'n_inundacion', 'n_vendaval',
       'n_creciente_subita', 'n_avenida_torrencial', 'n_otros',
       'densidad_poblacional', 'ingresos_tributarios',
       'inversion_gestion_riesgo', 'nbi_pct', 'elevacion_media',
       'pendiente_media', 'pct_pendiente_alta', 'ONI', 'fase_enso'],
      dtype='object')
Predicción nula: {'auc': np.float64(0.5), 'rmse_log': 1.335, 'winkler': 3.876}
Este modelo:     {'auc': np.float64(0.752), 'rmse_log': 1.241, 'winkler': 3.136}
Puntos automáticos en validación: 27.5 / 75


## Predicción de prueba_equipos

In [ ]:
modelo = entrenar(historial_entrenamiento(train))
prueba = pd.concat([test_pub, test_priv], ignore_index=True)
entrega = predecir(modelo, historial_congelado(train, prueba))
entrega["fecha"] = entrega["fecha"].dt.strftime("%Y-%m-%d")
entrega.head()

In [ ]:
COLUMNAS = ["DIVIPOLA", "fecha", "prob_evento", "personas_desplazadas_estimadas",
            "personas_desplazadas_q10", "personas_desplazadas_q90"]

def validar_entrega(e, test_pub=test_pub, test_priv=test_priv):
    errores = []
    faltan = [c for c in COLUMNAS if c not in e.columns]
    if faltan:
        return print("❌ Faltan columnas:", faltan)
    claves = pd.concat([test_pub, test_priv])[["DIVIPOLA", "fecha"]].assign(fecha=lambda d: pd.to_datetime(d.fecha))
    e2 = e.assign(DIVIPOLA=e.DIVIPOLA.astype(str).str.zfill(5), fecha=pd.to_datetime(e.fecha))
    if e2.duplicated(["DIVIPOLA", "fecha"]).any():
        errores.append("hay filas DIVIPOLA-fecha duplicadas")
    faltantes = len(claves.merge(e2, on=["DIVIPOLA", "fecha"], how="left", indicator=True).query("_merge == 'left_only'"))
    if faltantes:
        errores.append(f"faltan {faltantes} filas de las {len(claves)} esperadas")
    num = e2[COLUMNAS[2:]].apply(pd.to_numeric, errors="coerce")
    if num.isna().any().any():
        errores.append("hay valores vacíos o no numéricos")
    if not num.prob_evento.between(0, 1).all():
        errores.append("prob_evento debe estar entre 0 y 1")
    if (num[COLUMNAS[3:]] < 0).any().any():
        errores.append("las columnas de personas deben ser >= 0")
    if (num.personas_desplazadas_q10 > num.personas_desplazadas_q90).any():
        errores.append("personas_desplazadas_q10 debe ser <= personas_desplazadas_q90")
    print("✅ Entrega válida:", len(e), "filas" if not errores else "", *["\n❌ " + x for x in errores])
    return not errores

validar_entrega(entrega)

✅ Entrega válida: 42978 filas


True

### Guardar y descargar
Suban este archivo al formulario de entrega (el enlace está en la tabla de posiciones). Recuerden: máximo 1 entrega cada 30 minutos; cuenta la última válida antes de las 2:00 p. m.

In [ ]:
NOMBRE_EQUIPO = "Pato pato ganso"
archivo = f"entrega_{NOMBRE_EQUIPO}.csv"
entrega.to_csv(archivo, index=False)
try:
    from google.colab import files
    files.download(archivo)
except ImportError:
    print("Guardado en", os.path.abspath(archivo))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 5. Dashboard

In [ ]:
# Genera geovisor_dashboard.html con las predicciones de la entrega y los hallazgos del modelo.
# Uso en Colab, después de la sección 6 (entrega generada):   %run -i generar_dashboard.py
import json
import numpy as np
import pandas as pd

NOMBRE_EQUIPO = globals().get("NOMBRE_EQUIPO", "mi_equipo")
RUTA_COORDS = "coordenadas_municipios.csv"
SALIDA = "geovisor_dashboard.html"

DEPTOS = {"05": "Antioquia", "08": "Atlántico", "11": "Bogotá D.C.", "13": "Bolívar", "15": "Boyacá",
          "17": "Caldas", "18": "Caquetá", "19": "Cauca", "20": "Cesar", "23": "Córdoba",
          "25": "Cundinamarca", "27": "Chocó", "41": "Huila", "44": "La Guajira", "47": "Magdalena",
          "50": "Meta", "52": "Nariño", "54": "Norte de Santander", "63": "Quindío", "66": "Risaralda",
          "68": "Santander", "70": "Sucre", "73": "Tolima", "76": "Valle del Cauca", "81": "Arauca",
          "85": "Casanare", "86": "Putumayo", "88": "San Andrés", "91": "Amazonas", "94": "Guainía",
          "95": "Guaviare", "97": "Vaupés", "99": "Vichada"}

NOMBRES_VAR = {
    "depto_cat": "Departamento", "bioma_cat": "Bioma (WWF)", "mes_sin": "Mes (seno)", "mes_cos": "Mes (coseno)",
    "tasa_suave": "Tasa histórica suavizada", "sev_cond": "Tamaño típico de eventos", "hist_tasa": "Tasa histórica",
    "hist_tasa_mes": "Tasa histórica del mes", "log_viviendas": "Viviendas (exposición)",
    "precip_normal_mes": "Lluvia normal del mes", "precip_normal_anual": "Lluvia normal anual",
    "precip_estacional": "Estacionalidad de la lluvia", "dias_lluvia_normal_mes": "Días de lluvia normales",
    "log_densidad_poblacional": "Densidad poblacional", "log_ingresos_tributarios": "Ingresos tributarios",
    "elevacion_media": "Elevación media", "pendiente_media": "Pendiente media", "pct_pendiente_alta": "% pendiente alta",
    "susc_masa": "Susceptibilidad a deslizamiento", "susc_inund": "Susceptibilidad a inundación",
    "es_temporada_lluvias": "Temporada de lluvias", "lluvia_x_nina": "Lluvias × La Niña",
}

# ---- Predicciones ----
e = entrega.copy()
e["DIVIPOLA"] = e["DIVIPOLA"].astype(str).str.zfill(5)
e["fecha"] = pd.to_datetime(e["fecha"])
e["anio"] = e["fecha"].dt.year.astype(str)

coords = pd.read_csv(RUTA_COORDS, dtype={"DIVIPOLA": str})
coords["DIVIPOLA"] = coords["DIVIPOLA"].str.zfill(5)
munis = pd.DataFrame({"DIVIPOLA": sorted(e["DIVIPOLA"].unique())})
munis = munis.merge(coords[["DIVIPOLA", "municipio", "lat", "lon"]], on="DIVIPOLA", how="left")
munis["municipio"] = munis["municipio"].fillna(munis["DIVIPOLA"]).str.title()
munis["depto"] = munis["DIVIPOLA"].str[:2].map(DEPTOS).fillna("Otro")

umbral_alto = float(e["prob_evento"].quantile(0.90))     # "riesgo alto" = 10 % de mayor probabilidad
e["alto"] = (e["prob_evento"] >= umbral_alto).astype(int)


def resumen(df):
    g = df.groupby("DIVIPOLA").agg(p=("prob_evento", "mean"), alto=("alto", "sum"),
                                  pers=("personas_desplazadas_estimadas", "sum"),
                                  q90=("personas_desplazadas_q90", "max"))
    g = g.reindex(munis["DIVIPOLA"])
    return {"p": g["p"].round(4).tolist(), "alto": g["alto"].fillna(0).astype(int).tolist(),
            "pers": g["pers"].round(1).tolist(), "q90": g["q90"].round(0).tolist(),
            "meses": int(df["fecha"].nunique())}


periodos = {"Todo el horizonte": e}
periodos.update({a: e[e["anio"] == a] for a in sorted(e["anio"].unique())})
por_periodo = {k: resumen(v) for k, v in periodos.items()}

mensual = e.groupby("fecha").agg(p=("prob_evento", "mean"), alto=("alto", "sum")).reset_index()
dep = e.assign(depto=e["DIVIPOLA"].str[:2].map(DEPTOS).fillna("Otro")).groupby("depto").agg(
    p=("prob_evento", "mean"), alto=("alto", "sum")).sort_values("p", ascending=False).reset_index()

# ---- Importancia de variables (ganancia, modelo final) ----
importancia = []
try:
    clfs = globals()["modelo"].clfs
    imp = pd.Series(np.mean([c.booster_.feature_importance("gain") for c in clfs], axis=0),
                    index=clfs[0].feature_name_)
    imp = (imp / imp.sum()).sort_values(ascending=False)
    importancia = [{"var": NOMBRES_VAR.get(k, k), "imp": round(float(v) * 100, 1)} for k, v in imp.items()]
except Exception as ex:
    print("Sin importancia de variables:", ex)

# ---- Métricas de validación (sección 5) ----
m = globals().get("met", {"auc": 0.751, "rmse_log": 1.241, "winkler": 3.134})
n = globals().get("nula", {"auc": 0.5, "rmse_log": 1.335, "winkler": 3.876})
s_auc = max(0, (m["auc"] - 0.5) / 0.5)
s_rmse = min(1, max(0, 1 - m["rmse_log"] / n["rmse_log"]))
s_wink = min(1, max(0, 1 - m["winkler"] / n["winkler"]))
metricas = {"auc": float(m["auc"]), "rmse": float(m["rmse_log"]), "rmse_nulo": float(n["rmse_log"]),
            "wink": float(m["winkler"]), "wink_nulo": float(n["winkler"]),
            "pts_auc": 50 * s_auc, "pts_rmse": 20 * s_rmse, "pts_wink": 5 * s_wink,
            "pts": 50 * s_auc + 20 * s_rmse + 5 * s_wink}

DATOS = {
    "equipo": NOMBRE_EQUIPO,
    "n_mun": int(len(munis)), "n_filas": int(len(e)),
    "ini": e["fecha"].min().strftime("%Y-%m"), "fin": e["fecha"].max().strftime("%Y-%m"),
    "umbral_alto": round(umbral_alto, 3),
    "muni": {"cod": munis["DIVIPOLA"].tolist(), "nombre": munis["municipio"].tolist(),
             "depto": munis["depto"].tolist(), "lat": munis["lat"].round(4).tolist(),
             "lon": munis["lon"].round(4).tolist()},
    "periodos": list(periodos.keys()), "por_periodo": por_periodo,
    "mensual": {"fecha": mensual["fecha"].dt.strftime("%Y-%m").tolist(), "p": mensual["p"].round(4).tolist(),
                "alto": mensual["alto"].astype(int).tolist()},
    "depto": {"nombre": dep["depto"].tolist(), "p": dep["p"].round(4).tolist(), "alto": dep["alto"].astype(int).tolist()},
    "importancia": importancia[:15],
    "metricas": metricas,
    "progreso": [["Logística (base)", 18.6], ["LightGBM + historial suavizado", 24.1], ["+ Departamento", 25.0],
                 ["+ Regularización y 3 semillas", 25.7], ["− 6 variables de ruido", 26.1],
                 ["Magnitud = prob × tamaño", 26.3], ["+ Viviendas (exposición)", 26.6], ["+ Bioma WWF", 26.7],
                 ["+ Normales IDEAM", 27.5]],
}

PLANTILLA = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Geovisor de Riesgo de Desplazamiento</title>
__LIBRERIAS__
<style>
:root {
  color-scheme: light;
  --bg: #f4f3f0; --surface: #fcfcfb; --border: #e3e1dc;
  --text-primary: #0b0b0b; --text-secondary: #52514e; --text-muted: #77756f;
  --grid: #e8e6e1; --series: #2a78d6; --accent-soft: #e6effb;
  --seq-1: #cde2fb; --seq-2: #86b6ef; --seq-3: #3987e5; --seq-4: #1c5cab; --seq-5: #0d366b;
}
@media (prefers-color-scheme: dark) {
  :root:where(:not([data-theme="light"])) {
    color-scheme: dark;
    --bg: #121211; --surface: #1a1a19; --border: #2e2e2b;
    --text-primary: #ffffff; --text-secondary: #c3c2b7; --text-muted: #9a988f;
    --grid: #2e2e2b; --series: #3987e5; --accent-soft: #1d2a3b;
    --seq-1: #184f95; --seq-2: #2a78d6; --seq-3: #5598e7; --seq-4: #86b6ef; --seq-5: #cde2fb;
  }
}
:root[data-theme="dark"] {
  color-scheme: dark;
  --bg: #121211; --surface: #1a1a19; --border: #2e2e2b;
  --text-primary: #ffffff; --text-secondary: #c3c2b7; --text-muted: #9a988f;
  --grid: #2e2e2b; --series: #3987e5; --accent-soft: #1d2a3b;
  --seq-1: #184f95; --seq-2: #2a78d6; --seq-3: #5598e7; --seq-4: #86b6ef; --seq-5: #cde2fb;
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--text-primary);
  font: 15px/1.5 system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; }
.wrap { max-width: 1240px; margin: 0 auto; padding: 24px 16px 48px; }
header h1 { font-size: 26px; margin: 0 0 4px; letter-spacing: -0.01em; }
header p { margin: 0; color: var(--text-secondary); }
nav { display: flex; gap: 8px; flex-wrap: wrap; margin: 20px 0 8px; }
nav a { color: var(--text-secondary); text-decoration: none; padding: 6px 12px; border-radius: 999px;
  border: 1px solid var(--border); background: var(--surface); font-size: 14px; }
nav a:hover { color: var(--text-primary); border-color: var(--text-muted); }
section { margin-top: 32px; }
h2 { font-size: 19px; margin: 0 0 4px; }
.lead { color: var(--text-secondary); margin: 0 0 14px; max-width: 80ch; }
.grid { display: grid; gap: 16px; }
.kpis { grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); }
.two { grid-template-columns: 1fr 1fr; }
@media (max-width: 860px) { .two { grid-template-columns: 1fr; } }
.card { background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 16px; min-width: 0; }
.kpi .label { font-size: 13px; color: var(--text-secondary); }
.kpi .value { font-size: 30px; font-weight: 650; letter-spacing: -0.02em; margin-top: 2px; }
.kpi .value.small { font-size: 22px; padding: 4px 0; }
.kpi .sub { font-size: 13px; color: var(--text-muted); }
.card h3 { font-size: 15px; margin: 0 0 2px; }
.card .note { font-size: 13px; color: var(--text-muted); margin: 0 0 10px; }
.chart { position: relative; height: 300px; }
.chart.tall { height: 420px; }
#mapa { height: 480px; border-radius: 8px; }
.controls { display: flex; gap: 10px; align-items: center; flex-wrap: wrap; margin-bottom: 10px; }
select { font: inherit; padding: 6px 10px; border-radius: 8px; border: 1px solid var(--border);
  background: var(--surface); color: var(--text-primary); }
.legend { display: flex; gap: 6px; align-items: center; flex-wrap: wrap; font-size: 13px; color: var(--text-secondary); margin-top: 8px; }
.legend .sw { width: 14px; height: 14px; border-radius: 4px; display: inline-block; }
table { width: 100%; border-collapse: collapse; font-size: 14px; }
th, td { text-align: left; padding: 8px 10px; border-bottom: 1px solid var(--border); vertical-align: top; }
th { color: var(--text-secondary); font-weight: 600; font-size: 13px; }
td.num, th.num { text-align: right; font-variant-numeric: tabular-nums; }
.scroll { overflow-x: auto; }
.find { display: grid; gap: 16px; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); }
.find .card h3 { font-size: 16px; }
.find .card p { margin: 6px 0 0; color: var(--text-secondary); }
.tag { display: inline-block; font-size: 12px; padding: 2px 8px; border-radius: 999px; background: var(--accent-soft); color: var(--text-primary); }
.flow { display: grid; gap: 10px; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); counter-reset: paso; }
.flow .card { position: relative; padding-top: 34px; }
.flow .card::before { counter-increment: paso; content: counter(paso); position: absolute; top: 12px; left: 16px;
  width: 22px; height: 22px; border-radius: 50%; background: var(--series); color: #fff; font-size: 12px;
  display: grid; place-items: center; font-weight: 600; }
.flow .card p { margin: 4px 0 0; color: var(--text-secondary); font-size: 14px; }
footer { margin-top: 40px; color: var(--text-muted); font-size: 13px; }
.leaflet-container { background: var(--surface); font: inherit; }
</style>
</head>
<body>
<div class="wrap">
<header>
  <h1>Riesgo de desplazamiento por desastres climáticos</h1>
  <p id="subtitulo"></p>
  <nav>
    <a href="#resultados">Resultados</a><a href="#mapa-sec">Mapa</a><a href="#tiempo">En el tiempo</a>
    <a href="#modelo">Cómo funciona</a><a href="#hallazgos">Hallazgos</a><a href="#validacion">Validación</a>
    <a href="#datos">Datos y replicabilidad</a>
  </nav>
</header>

<section id="resultados">
  <div class="grid kpis" id="kpis"></div>
</section>

<section id="mapa-sec">
  <h2>¿Dónde está el riesgo?</h2>
  <p class="lead">Probabilidad promedio de un evento climático con afectación en cada municipio. Pase el cursor sobre un punto para ver el detalle.</p>
  <div class="grid two">
    <div class="card">
      <div class="controls">
        <label for="periodo">Periodo</label>
        <select id="periodo"></select>
      </div>
      <div id="mapa" role="img" aria-label="Mapa de probabilidad de evento por municipio"></div>
      <div class="legend" id="leyenda-mapa"></div>
    </div>
    <div class="card">
      <h3>Municipios con mayor probabilidad</h3>
      <p class="note" id="nota-top"></p>
      <div class="scroll"><table id="tabla-top"></table></div>
    </div>
  </div>
</section>

<section id="tiempo">
  <h2>¿Cuándo?</h2>
  <p class="lead">Evolución mensual del riesgo en todo el horizonte de predicción.</p>
  <div class="grid two">
    <div class="card"><h3>Probabilidad promedio nacional</h3><p class="note">Promedio de los municipios en cada mes.</p>
      <div class="chart"><canvas id="g-mensual"></canvas></div></div>
    <div class="card"><h3>Municipios en riesgo alto por mes</h3><p class="note" id="nota-alto"></p>
      <div class="chart"><canvas id="g-alto"></canvas></div></div>
  </div>
  <div class="card" style="margin-top:16px"><h3>Probabilidad promedio por departamento</h3>
    <p class="note">Todo el horizonte de predicción.</p>
    <div class="chart tall"><canvas id="g-depto"></canvas></div></div>
</section>

<section id="modelo">
  <h2>Cómo funciona el modelo</h2>
  <p class="lead">LightGBM (árboles de decisión en secuencia) entrenado con el historial de 2018–2022 y variables de territorio, clima y exposición.</p>
  <div class="flow">
    <div class="card"><h3>Historial sin fuga</h3><p>Cada mes solo ve los meses anteriores del municipio. Las tasas se suavizan hacia el promedio nacional cuando hay pocos datos.</p></div>
    <div class="card"><h3>Variables</h3><p>Historial, territorio, socioeconomía, exposición (viviendas), clima propio de cada municipio (IDEAM) y región.</p></div>
    <div class="card"><h3>Probabilidad</h3><p>Promedio de 3 modelos LightGBM regularizados, entrenados con semillas distintas.</p></div>
    <div class="card"><h3>Magnitud</h3><p>Personas esperadas = probabilidad × tamaño típico de los eventos del municipio.</p></div>
    <div class="card"><h3>Intervalo 80 %</h3><p>Modelos de cuantiles 10 % y 90 %. Si el riesgo es ínfimo, el intervalo es [0, 0].</p></div>
  </div>
  <div class="grid two" style="margin-top:16px">
    <div class="card"><h3>Qué pesa más en la predicción</h3><p class="note">Importancia de cada variable en el modelo final (% de ganancia).</p>
      <div class="chart tall"><canvas id="g-imp"></canvas></div></div>
    <div class="card"><h3>Cómo subió el puntaje</h3><p class="note">Puntos automáticos en validación (máx. 75). Cada paso se mantuvo solo si mejoraba.</p>
      <div class="chart tall"><canvas id="g-prog"></canvas></div></div>
  </div>
</section>

<section id="hallazgos">
  <h2>Hallazgos</h2>
  <div class="find">
    <div class="card"><span class="tag">Ubicación</span><h3>La región pesa más que todo</h3><p>El departamento fue la mejora más estable: cada región tiene su propio patrón de eventos y su propia respuesta al clima.</p></div>
    <div class="card"><span class="tag">Clima local</span><h3>Cada municipio tiene su temporada</h3><p>Usar la lluvia normal de 30 años (IDEAM) de cada municipio subió 0.8 puntos. Una temporada de lluvias nacional fija no describe a La Guajira ni al Chocó.</p></div>
    <div class="card"><span class="tag">Exposición</span><h3>Más viviendas, más afectación</h3><p>El total de viviendas del censo aportó información que el modelo no tenía: cuánta gente puede verse afectada.</p></div>
    <div class="card"><span class="tag">ENSO</span><h3>El ONI global aporta poco por sí solo</h3><p>Es igual para todos los municipios en un mes y el periodo de prueba tiene otro clima (La Niña: 42 % de los meses de entrenamiento vs. 18 % de la prueba).</p></div>
  </div>
</section>

<section id="validacion">
  <h2>Validación</h2>
  <p class="lead">Entrenado hasta 2021-09 y evaluado de 2021-10 a 2022-09, meses que el modelo no vio. Métricas oficiales del reto, comparadas con la predicción nula.</p>
  <div class="grid two">
    <div class="card scroll"><table id="tabla-metricas"></table>
      <p class="note" style="margin-top:12px">Las personas estimadas son un valor esperado en escala logarítmica: combinan la probabilidad de que no pase nada con el tamaño de los eventos. Para decidir, use la probabilidad y el escenario alto (q90).</p></div>
    <div class="card scroll"><h3>Lo que se probó y se descartó</h3>
      <table>
        <tr><th>Prueba</th><th>Resultado</th></tr>
        <tr><td>Rezagos del ONI</td><td>Bajó: el modelo memorizaba fechas</td></tr>
        <tr><td>Tasa de los últimos 12 meses</td><td>Bajó: en la prueba esos meses quedan lejos</td></tr>
        <tr><td>Modelo propio de severidad</td><td>Bajó: muy pocos eventos para aprender el tamaño</td></tr>
        <tr><td>Cruces riesgo × mes, departamento × mes</td><td>Bajó: LightGBM ya hace esos cruces</td></tr>
        <tr><td>Materiales de vivienda, región natural, % rural</td><td>Igual: información ya presente</td></tr>
        <tr><td>Humedad y balance hídrico (IDEAM)</td><td>Igual: repiten lo que dice la lluvia</td></tr>
      </table></div>
  </div>
</section>

<section id="datos">
  <h2>Datos y replicabilidad</h2>
  <div class="card scroll">
    <table>
      <tr><th>Variable</th><th>Fuente en este modelo</th><th>Proxy abierto y global</th></tr>
      <tr><td>Historial de eventos y afectados</td><td>UNGRD (datos del reto)</td><td>DesInventar (UNDRR) / EM-DAT</td></tr>
      <tr><td>Densidad poblacional</td><td>TerriData (DNP)</td><td>WorldPop</td></tr>
      <tr><td>Ingresos tributarios</td><td>TerriData (DNP)</td><td>PIB subnacional en grilla (Kummu et al., 2018)</td></tr>
      <tr><td>NBI (en susceptibilidad)</td><td>Censo 2018 (DANE)</td><td>GRDI, NASA SEDAC</td></tr>
      <tr><td>Elevación y pendiente</td><td>Copernicus DEM</td><td>Copernicus DEM / SRTM</td></tr>
      <tr><td>ONI</td><td>NOAA CPC</td><td>NOAA CPC</td></tr>
      <tr><td>Viviendas (exposición)</td><td>Censo 2018 (DANE)</td><td>Google Open Buildings / GHSL / WorldPop</td></tr>
      <tr><td>Lluvia normal por mes</td><td>IDEAM, Normales 1991–2020</td><td>WorldClim 2.1 / climatología CHIRPS</td></tr>
      <tr><td>Bioma</td><td>WWF Terrestrial Ecoregions</td><td>RESOLVE Ecoregions 2017</td></tr>
      <tr><td>Departamento</td><td>DIVIPOLA (DANE)</td><td>GADM nivel 1</td></tr>
    </table>
  </div>
</section>

<footer id="pie"></footer>
</div>

<script>
const D = __DATOS__;
const GEO = __GEOJSON__;
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const fmtP = v => (v == null || isNaN(v)) ? "—" : (v * 100).toFixed(1) + " %";
const fmtN = v => (v == null || isNaN(v)) ? "—" : Math.round(v).toLocaleString("es-CO");

document.getElementById("subtitulo").textContent =
  `Predicciones para ${D.n_mun.toLocaleString("es-CO")} municipios, ${D.ini} a ${D.fin} · Equipo ${D.equipo}`;
document.getElementById("pie").textContent =
  `Hackathón PNUD Colombia · Modelo LightGBM · ${D.n_filas.toLocaleString("es-CO")} predicciones municipio-mes`;

// ---- KPIs ----
const M = D.metricas;
const kpis = [
  ["Puntos en validación", M.pts.toFixed(1), "de 75 automáticos"],
  ["AUC (detección de riesgo)", M.auc.toFixed(3), "0.5 sería azar"],
  ["Municipios", D.n_mun.toLocaleString("es-CO"), `${D.n_filas.toLocaleString("es-CO")} filas municipio-mes`],
  ["Horizonte", `${D.ini} a ${D.fin}`, "prueba_equipos + prueba_oculta", "small"],
];
document.getElementById("kpis").innerHTML = kpis.map(k =>
  `<div class="card kpi"><div class="label">${k[0]}</div><div class="value${k[3] ? " small" : ""}">${k[1]}</div><div class="sub">${k[2]}</div></div>`).join("");

// ---- Mapa ----
const sel = document.getElementById("periodo");
sel.innerHTML = D.periodos.map(p => `<option>${p}</option>`).join("");
const mapa = L.map("mapa", { preferCanvas: true, zoomSnap: 0.5 }).setView([4.6, -73.8], 5.5);
let capaBase = null, capaDeptos = null;
mapa.createPane("deptos"); mapa.getPane("deptos").style.zIndex = 350;
const rendDeptos = L.svg({ pane: "deptos" });
function base() {
  const oscuro = matchMedia("(prefers-color-scheme: dark)").matches && document.documentElement.dataset.theme !== "light"
               || document.documentElement.dataset.theme === "dark";
  if (capaBase) mapa.removeLayer(capaBase);
  if (capaDeptos) mapa.removeLayer(capaDeptos);
  const estilo = oscuro ? "World_Dark_Gray_Base" : "World_Light_Gray_Base";
  capaBase = L.tileLayer(`https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/${estilo}/MapServer/tile/{z}/{y}/{x}`,
    { attribution: "Tiles &copy; Esri", maxZoom: 16 }).addTo(mapa);
  if (GEO) {
    capaDeptos = L.geoJSON(GEO, { pane: "deptos", renderer: rendDeptos, interactive: false,
      style: { color: css("--text-muted"), weight: 1, fillColor: css("--surface"), fillOpacity: 0.6 } }).addTo(mapa);
  }
  setTimeout(() => mapa.invalidateSize(), 300);
}
base();
const capaPuntos = L.layerGroup().addTo(mapa);
let cortes = [];
function color(v) {
  const s = [css("--seq-1"), css("--seq-2"), css("--seq-3"), css("--seq-4"), css("--seq-5")];
  for (let i = 0; i < cortes.length; i++) if (v < cortes[i]) return s[i];
  return s[4];
}
function pintar() {
  const R = D.por_periodo[sel.value], mu = D.muni;
  const vals = R.p.filter(v => v != null).slice().sort((a, b) => a - b);
  cortes = [0.2, 0.4, 0.6, 0.8].map(q => vals[Math.floor(q * (vals.length - 1))]);
  capaPuntos.clearLayers();
  const ring = css("--surface");
  mu.cod.forEach((c, i) => {
    if (mu.lat[i] == null || R.p[i] == null) return;
    L.circleMarker([mu.lat[i], mu.lon[i]], { radius: 5, weight: 1, color: ring, fillColor: color(R.p[i]), fillOpacity: 0.9 })
      .bindTooltip(`<b>${mu.nombre[i]}</b> (${mu.depto[i]})<br>Probabilidad promedio: ${fmtP(R.p[i])}<br>` +
                   `Meses en riesgo alto: ${R.alto[i]} de ${R.meses}<br>Escenario alto (q90 máx.): ${fmtN(R.q90[i])} personas`)
      .addTo(capaPuntos);
  });
  const s = [1, 2, 3, 4, 5].map(k => css(`--seq-${k}`));
  const lim = [vals[0], ...cortes, vals[vals.length - 1]];
  document.getElementById("leyenda-mapa").innerHTML = "Probabilidad promedio:" + s.map((c, i) =>
    `<span class="sw" style="background:${c}"></span>${fmtP(lim[i])}–${fmtP(lim[i + 1])}`).join(" ");
  // tabla top
  const idx = mu.cod.map((_, i) => i).filter(i => R.p[i] != null).sort((a, b) => R.p[b] - R.p[a]).slice(0, 15);
  document.getElementById("nota-top").textContent = `${sel.value}. Riesgo alto = probabilidad ≥ ${fmtP(D.umbral_alto)} (el 10 % más alto de todas las predicciones).`;
  document.getElementById("tabla-top").innerHTML =
    `<tr><th>#</th><th>Municipio</th><th>Departamento</th><th class="num">Prob. promedio</th><th class="num">Meses riesgo alto</th></tr>` +
    idx.map((i, k) => `<tr><td>${k + 1}</td><td>${mu.nombre[i]}</td><td>${mu.depto[i]}</td>` +
      `<td class="num">${fmtP(R.p[i])}</td><td class="num">${R.alto[i]} / ${R.meses}</td></tr>`).join("");
}
sel.addEventListener("change", pintar);
pintar();

// ---- Gráficos ----
const graficos = [];
function opcionesBase(extra = {}) {
  const t2 = css("--text-secondary"), g = css("--grid");
  return Object.assign({
    responsive: true, maintainAspectRatio: false, animation: false,
    interaction: { mode: "index", intersect: false },
    plugins: { legend: { display: false }, tooltip: { displayColors: false } },
    scales: {
      x: { ticks: { color: t2, maxRotation: 0, autoSkip: true }, grid: { display: false }, border: { color: g } },
      y: { ticks: { color: t2 }, grid: { color: g }, border: { display: false }, beginAtZero: true },
    },
  }, extra);
}
function dibujar() {
  graficos.forEach(g => g.destroy()); graficos.length = 0;
  const azul = css("--series");
  graficos.push(new Chart(document.getElementById("g-mensual"), {
    type: "line",
    data: { labels: D.mensual.fecha, datasets: [{ data: D.mensual.p.map(v => v * 100), borderColor: azul, backgroundColor: azul,
      borderWidth: 2, pointRadius: 0, pointHoverRadius: 5, tension: 0.25 }] },
    options: opcionesBase({ plugins: { legend: { display: false }, tooltip: { displayColors: false,
      callbacks: { label: c => `Probabilidad promedio: ${c.parsed.y.toFixed(1)} %` } } },
      scales: Object.assign(opcionesBase().scales, { y: Object.assign(opcionesBase().scales.y, { ticks: { color: css("--text-secondary"), callback: v => v + " %" } }) }) }),
  }));
  graficos.push(new Chart(document.getElementById("g-alto"), {
    type: "bar",
    data: { labels: D.mensual.fecha, datasets: [{ data: D.mensual.alto, backgroundColor: azul, borderRadius: 4, borderSkipped: "start", categoryPercentage: 0.8, barPercentage: 0.9 }] },
    options: opcionesBase({ plugins: { legend: { display: false }, tooltip: { displayColors: false,
      callbacks: { label: c => `${c.parsed.y} municipios en riesgo alto` } } } }),
  }));
  const nd = D.depto.nombre.length;
  graficos.push(new Chart(document.getElementById("g-depto"), {
    type: "bar",
    data: { labels: D.depto.nombre, datasets: [{ data: D.depto.p.map(v => v * 100), backgroundColor: azul, borderRadius: 4, borderSkipped: "start", categoryPercentage: 0.8, barPercentage: 0.9 }] },
    options: opcionesBase({ plugins: { legend: { display: false }, tooltip: { displayColors: false,
      callbacks: { label: c => `Probabilidad promedio: ${c.parsed.y.toFixed(1)} %` } } },
      scales: Object.assign(opcionesBase().scales, {
        x: { ticks: { color: css("--text-secondary"), autoSkip: false, maxRotation: 60, minRotation: 45, font: { size: 11 } }, grid: { display: false } },
        y: Object.assign(opcionesBase().scales.y, { ticks: { color: css("--text-secondary"), callback: v => v + " %" } }) }) }),
  }));
  if (D.importancia.length) {
    graficos.push(new Chart(document.getElementById("g-imp"), {
      type: "bar",
      data: { labels: D.importancia.map(d => d.var), datasets: [{ data: D.importancia.map(d => d.imp), backgroundColor: azul, borderRadius: 4, borderSkipped: "start", categoryPercentage: 0.8, barPercentage: 0.85 }] },
      options: opcionesBase({ indexAxis: "y", interaction: { mode: "nearest", axis: "y", intersect: false },
        plugins: { legend: { display: false }, tooltip: { displayColors: false, callbacks: { label: c => `${c.parsed.x.toFixed(1)} % de la ganancia` } } },
        scales: { x: { ticks: { color: css("--text-secondary"), callback: v => v + " %" }, grid: { color: css("--grid") }, border: { display: false }, beginAtZero: true },
                  y: { ticks: { color: css("--text-primary"), autoSkip: false }, grid: { display: false } } } }),
    }));
  } else {
    document.getElementById("g-imp").parentElement.innerHTML = `<p class="note">Importancia no disponible.</p>`;
  }
  graficos.push(new Chart(document.getElementById("g-prog"), {
    type: "bar",
    data: { labels: D.progreso.map(p => p[0]), datasets: [{ data: D.progreso.map(p => p[1]), backgroundColor: azul, borderRadius: 4, borderSkipped: "start", categoryPercentage: 0.8, barPercentage: 0.85 }] },
    options: opcionesBase({ indexAxis: "y", interaction: { mode: "nearest", axis: "y", intersect: false },
      plugins: { legend: { display: false }, tooltip: { displayColors: false, callbacks: { label: c => `${c.parsed.x.toFixed(1)} puntos` } } },
      layout: { padding: { right: 36 } },
      scales: { x: { min: 0, max: 30, ticks: { color: css("--text-secondary") }, grid: { color: css("--grid") }, border: { display: false } },
                y: { ticks: { color: css("--text-primary"), autoSkip: false }, grid: { display: false } } } }),
    plugins: [{ id: "etiquetas", afterDatasetsDraw(ch) {
      const meta = ch.getDatasetMeta(0), ctx = ch.ctx, n = meta.data.length;
      ctx.save(); ctx.font = "600 12px system-ui, sans-serif"; ctx.fillStyle = css("--text-primary"); ctx.textBaseline = "middle";
      [0, n - 1].forEach(i => { const b = meta.data[i]; ctx.fillText(D.progreso[i][1].toFixed(1), b.x + 6, b.y); });
      ctx.restore(); } }],
  }));
}
dibujar();

// ---- Tabla de métricas ----
document.getElementById("tabla-metricas").innerHTML =
  `<tr><th>Componente</th><th class="num">Modelo</th><th class="num">Nula</th><th class="num">Puntos</th></tr>` +
  `<tr><td>Detección de riesgo (AUC)</td><td class="num">${M.auc.toFixed(3)}</td><td class="num">0.500</td><td class="num">${M.pts_auc.toFixed(1)} / 50</td></tr>` +
  `<tr><td>Magnitud (RMSE log)</td><td class="num">${M.rmse.toFixed(3)}</td><td class="num">${M.rmse_nulo.toFixed(3)}</td><td class="num">${M.pts_rmse.toFixed(1)} / 20</td></tr>` +
  `<tr><td>Incertidumbre (Winkler 80 %)</td><td class="num">${M.wink.toFixed(3)}</td><td class="num">${M.wink_nulo.toFixed(3)}</td><td class="num">${M.pts_wink.toFixed(1)} / 5</td></tr>` +
  `<tr><td><b>Total automático</b></td><td></td><td></td><td class="num"><b>${M.pts.toFixed(1)} / 75</b></td></tr>`;

matchMedia("(prefers-color-scheme: dark)").addEventListener("change", () => { base(); pintar(); dibujar(); });
new MutationObserver(() => { base(); pintar(); dibujar(); }).observe(document.documentElement, { attributes: true, attributeFilter: ["data-theme"] });
</script>
</body>
</html>
"""

# ---- Librerías y contornos de departamentos (igual que el geovisor anterior) ----
URLS = {"leaflet_css": "https://unpkg.com/leaflet@1.9.4/dist/leaflet.css",
        "leaflet_js": "https://unpkg.com/leaflet@1.9.4/dist/leaflet.js",
        "chart_js": "https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"}
URL_GEOJSON = ("https://gist.githubusercontent.com/john-guerra/43c7656821069d00dcbc/raw/"
               "be6a6e239cd5b5b803c6e7c2ec405b793a9064dd/Colombia.geo.json")
libs = {}
try:
    import requests
except ImportError:
    requests = None
for nombre, url in URLS.items():
    texto = globals().get(nombre)                      # si ya se descargaron en el notebook, se reutilizan
    if not texto and requests is not None:
        try:
            texto = requests.get(url, timeout=30).text
        except Exception as ex:
            print(f"No se pudo descargar {nombre}: {ex}")
    libs[nombre] = texto
if all(libs.values()):                                 # incrustadas: el HTML funciona sin depender de CDN
    librerias = (f"<style>{libs['leaflet_css']}</style>\n<script>{libs['leaflet_js']}</script>\n"
                 f"<script>{libs['chart_js']}</script>")
else:                                                  # respaldo: cargar desde CDN al abrir
    librerias = (f'<link rel="stylesheet" href="{URLS["leaflet_css"]}">\n<script src="{URLS["leaflet_js"]}"></script>\n'
                 f'<script src="{URLS["chart_js"]}"></script>')

geojson = globals().get("geojson_deptos")
if geojson is None and requests is not None:
    try:
        geojson = requests.get(URL_GEOJSON, timeout=30).json()
    except Exception as ex:
        print("No se pudo descargar el GeoJSON de departamentos (el mapa funciona sin contornos):", ex)

html = (PLANTILLA.replace("__LIBRERIAS__", librerias)
        .replace("__GEOJSON__", json.dumps(geojson, ensure_ascii=False))
        .replace("__DATOS__", json.dumps(DATOS, ensure_ascii=False)))
with open(SALIDA, "w", encoding="utf-8") as f:
    f.write(html)
print(f"Dashboard guardado: {SALIDA} ({len(html) / 1024:.0f} KB)")
try:
    from google.colab import files
    files.download(SALIDA)
except ImportError:
    pass

Dashboard guardado: geovisor_dashboard.html (1938 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 6. Bibliografía

Departamento Administrativo Nacional de Estadística [DANE]. (2018). Censo Nacional de Población y Vivienda 2018: viviendas por municipio y por área (cabecera, centro poblado y rural disperso) [Conjunto de datos]. https://www.dane.gov.co/index.php/estadisticas-por-tema/demografia-y-poblacion/censo-nacional-de-poblacion-y-vivenda-2018

Instituto de Hidrología, Meteorología y Estudios Ambientales [IDEAM]. (s.f.). Normales climatológicas de Colombia, periodo 1991–2020 [Conjunto de datos]. Portal de Datos Abiertos de Colombia. https://www.datos.gov.co

Departamento Administrativo Nacional de Estadística [DANE]. (s.f.). DIVIPOLA: códigos de municipios con coordenadas geográficas [Conjunto de datos]. Portal de Datos Abiertos de Colombia. https://www.datos.gov.co/api/views/gdxc-w37w

Anthropic. (2026). Claude (Versión claude-opus-5-5) [Modelo de lenguaje de gran tamaño]. https://claude.ai